# Projeto (Seção 8) — solução do instrutor

## Especificação

Novo campo `registration_date` (após `registration_country`, `date | None = None`, não anterior a `manufactured_date`, JSON `YYYY/MM/DD`) e
tipo anotado `Country` que valida contra o dicionário `countries` (busca por minúsculas/sem espaços) e substitui pelo nome formatado.
Enunciado em [`07_projeto`](07_projeto.ipynb).

In [1]:
from datetime import date
from enum import Enum
from typing import Annotated, TypeVar
from uuid import UUID, uuid4
from pydantic import BaseModel, ConfigDict, Field, UUID4, field_serializer
from pydantic.alias_generators import to_camel


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


BoundedString = Annotated[str, Field(min_length=2, max_length=50)]

T = TypeVar('T')

BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]

In [2]:
countries = {
    "australia": ("Australia", "AUS"),
    "canada": ("Canada", "CAN"),
    "china": ("China", "CHN"),
    "france": ("France", "FRA"),
    "germany": ("Germany", "DEU"),
    "india": ("India", "IND"),
    "mexico": ("Mexico", "MEX"),
    "norway": ("Norway", "NOR"),
    "pakistan": ("Pakistan", "PAK"),
    "san marino": ("San Marino", "SMR"),
    "sanmarino": ("San Marino", "SMR"),
    "spain": ("Spain", "ESP"),
    "sweden": ("Sweden", "SWE"),
    "united kingdom": ("United Kingdom", "GBR"),
    "uk": ("United Kingdom", "GBR"),
    "great britain": ("United Kingdom", "GBR"),
    "britain": ("United Kingdom", "GBR"),
    "us": ("United States of America", "USA"),
    "united states": ("United States of America", "USA"),
    "usa": ("United States of America", "USA"),
}

In [3]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "us",
    "registrationDate": "2023-06-01",
    "licensePlate": "AAA-BBB"
}

expected_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4 Competition xDrive',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'United States of America',
    'registrationDate': date(2023, 6, 1),
    'licensePlate': 'AAA-BBB'
}

expected_json_by_alias = '{"id":"c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7","manufacturer":"BMW","seriesName":"M4 Competition xDrive","type":"Convertible","isElectric":false,"manufacturedDate":"2023/01/01","baseMSRPUSD":93300.0,"topFeatures":["6 cylinders","all-wheel drive","convertible"],"vin":"1234567890","numberOfDoors":2,"registrationCountry":"United States of America","registrationDate":"2023/06/01","licensePlate":"AAA-BBB"}'

## Solução

Vamos escrever uma função de busca que devolve nome e código do país a partir da entrada — com busca sem diferenciar maiúsculas/minúsculas
e removendo espaços (caso ainda não tenham sido removidos).

Pré-calculamos a lista de nomes válidos (a partir do dicionário `countries`), para devolvê-la quando a validação do país falhar. Esse código
não vai dentro do validador para não pagar o custo de regerar a lista a cada execução.

In [4]:
valid_country_names = sorted(countries.keys())
print(f"valid_country_names: {valid_country_names!r}")

valid_country_names: ['australia', 'britain', 'canada', 'china', 'france', 'germany', 'great britain', 'india', 'mexico', 'norway', 'pakistan', 'san marino', 'sanmarino', 'spain', 'sweden', 'uk', 'united kingdom', 'united states', 'us', 'usa']


In [5]:
def lookup_country(name: str) -> tuple[str, str]:
    name = name.strip().casefold()

    try:
        return countries[name]
    except KeyError:
        raise ValueError(
            "Unknown country name. "
            f"Country name must be one of: {','.join(valid_country_names)}"
        )

Agora o tipo anotado para países:

In [6]:
from pydantic import AfterValidator

Country = Annotated[str, AfterValidator(lambda name: lookup_country(name)[0])]

E o usamos no modelo:

In [7]:
class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel,
    )

    id_: UUID4 | None = Field(alias="id", default_factory=uuid4)
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate", ge=date(1980, 1, 1))
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD",
        serialization_alias="baseMSRPUSD"
    )
    top_features: BoundedList[BoundedString] | None = None
    vin: BoundedString
    number_of_doors: int = Field(
        default=4,
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
    )
    registration_country: Country | None = None
    registration_date: date | None = None
    license_plate: BoundedString | None = None

    @field_serializer("manufactured_date", "registration_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

Ainda falta o validador de `registration_date`, que verifica se a data não é anterior a `manufactured_date`.

Adicionando direto no modelo:

In [8]:
from pydantic import field_validator, ValidationInfo


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=to_camel,
    )

    id_: UUID4 | None = Field(alias="id", default_factory=uuid4)
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType = Field(alias="type")
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate", ge=date(1980, 1, 1))
    base_msrp_usd: float = Field(
        validation_alias="msrpUSD",
        serialization_alias="baseMSRPUSD"
    )
    top_features: BoundedList[BoundedString] | None = None
    vin: BoundedString
    number_of_doors: int = Field(
        default=4,
        validation_alias="doors",
        ge=2,
        le=4,
        multiple_of=2,
    )
    registration_country: Country | None = None
    registration_date: date | None = None
    license_plate: BoundedString | None = None

    @field_serializer("manufactured_date", "registration_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

    @field_validator("registration_date")
    @classmethod
    def validate_registration_date(cls, value:date, values: ValidationInfo):
        data = values.data
        if "manufactured_date" in data and data["manufactured_date"] > value:
            raise ValueError("Automobile cannot be registered prior to manufacture date.")
        return value

Desserializando e serializando:

In [9]:
car = Automobile.model_validate(data)
print(f"car: {car!r}")

car: Automobile(id_=UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'), manufacturer='BMW', series_name='M4 Competition xDrive', type_=<AutomobileType.convertible: 'Convertible'>, is_electric=False, manufactured_date=datetime.date(2023, 1, 1), base_msrp_usd=93300.0, top_features=['6 cylinders', 'all-wheel drive', 'convertible'], vin='1234567890', number_of_doors=2, registration_country='United States of America', registration_date=datetime.date(2023, 6, 1), license_plate='AAA-BBB')


In [10]:
assert car.model_dump(by_alias=True) == expected_by_alias

In [11]:
assert car.model_dump_json(by_alias=True) == expected_json_by_alias

Testando as validações: data de registro anterior à fabricação e país não suportado:

In [12]:
bad_data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "Lunar Colony",
    "registrationDate": "2022-06-01",
    "licensePlate": "AAA-BBB"
}

In [13]:
from pydantic import ValidationError

try:
    Automobile.model_validate(bad_data)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Automobile
registrationCountry
  Value error, Unknown country name. Country name must be one of: australia,britain,canada,china,france,germany,great britain,india,mexico,norway,pakistan,san marino,sanmarino,spain,sweden,uk,united kingdom,united states,us,usa [type=value_error, input_value='Lunar Colony', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
registrationDate
  Value error, Automobile cannot be registered prior to manufacture date. [type=value_error, input_value='2022-06-01', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


Um parêntese: até aqui sempre imprimimos as exceções de validação do Pydantic. Mas elas têm o método especial `.json()`, que devolve todos
os erros como um objeto JSON. Frameworks como o FastAPI usam isso para informar ao cliente da API o que falhou.

O método aceita `indent` — normalmente não usamos ao devolver JSON numa API (queremos os dados compactos), mas aqui ajuda a ler.

In [14]:
try:
    Automobile.model_validate(bad_data)
except ValidationError as ex:
    exceptions = ex.json(indent=2)

In [15]:
print(f"exceptions: {exceptions}")

exceptions: [
  {
    "type": "value_error",
    "loc": [
      "registrationCountry"
    ],
    "msg": "Value error, Unknown country name. Country name must be one of: australia,britain,canada,china,france,germany,great britain,india,mexico,norway,pakistan,san marino,sanmarino,spain,sweden,uk,united kingdom,united states,us,usa",
    "input": "Lunar Colony",
    "ctx": {
      "error": "Unknown country name. Country name must be one of: australia,britain,canada,china,france,germany,great britain,india,mexico,norway,pakistan,san marino,sanmarino,spain,sweden,uk,united kingdom,united states,us,usa"
    },
    "url": "https://errors.pydantic.dev/2.13/v/value_error"
  },
  {
    "type": "value_error",
    "loc": [
      "registrationDate"
    ],
    "msg": "Value error, Automobile cannot be registered prior to manufacture date.",
    "input": "2022-06-01",
    "ctx": {
      "error": "Automobile cannot be registered prior to manufacture date."
    },
    "url": "https://errors.pydantic.de

Também dá para obter como objeto Python:

In [16]:
try:
    Automobile.model_validate(bad_data)
except ValidationError as ex:
    exceptions = ex.errors()

In [17]:
print(f"type(exceptions): {type(exceptions)!r}")

type(exceptions): <class 'list'>


In [18]:
from pprint import pprint

pprint(exceptions)

[{'ctx': {'error': ValueError('Unknown country name. Country name must be one of: australia,britain,canada,china,france,germany,great britain,india,mexico,norway,pakistan,san marino,sanmarino,spain,sweden,uk,united kingdom,united states,us,usa')},
  'input': 'Lunar Colony',
  'loc': ('registrationCountry',),
  'msg': 'Value error, Unknown country name. Country name must be one of: '
         'australia,britain,canada,china,france,germany,great '
         'britain,india,mexico,norway,pakistan,san '
         'marino,sanmarino,spain,sweden,uk,united kingdom,united states,us,usa',
  'type': 'value_error',
  'url': 'https://errors.pydantic.dev/2.13/v/value_error'},
 {'ctx': {'error': ValueError('Automobile cannot be registered prior to manufacture date.')},
  'input': '2022-06-01',
  'loc': ('registrationDate',),
  'msg': 'Value error, Automobile cannot be registered prior to manufacture '
         'date.',
  'type': 'value_error',
  'url': 'https://errors.pydantic.dev/2.13/v/value_error'}]

> 📝 **Nota de revisão — bug nesta solução:** o validador compara `data["manufactured_date"] > value` sem tratar `value is None`. Como o
> modelo tem `validate_default=True`, o default `None` de `registration_date` **também passa pelo validador** — e `date > None` levanta
> `TypeError`, que o Pydantic **não** converte em `ValidationError` (vimos isso na aula 02). Resultado: qualquer carro **sem**
> `registrationDate` derruba a desserialização com uma exceção crua. Prova:

In [19]:
data_without_registration = {k: v for k, v in data.items() if k != "registrationDate"}

try:
    Automobile.model_validate(data_without_registration)
except Exception as ex:
    print(f"{type(ex).__name__}: {ex}")

TypeError: '>' not supported between instances of 'datetime.date' and 'NoneType'


A correção é tratar o `None` antes de comparar (`if value is not None and ...`). A solução comentada faz isso e testa o caso.

> 📝 **Nota de revisão 2:** o `id_` continua `UUID4 | None` (a regressão da seção 7).